# Local model prediction

This notebook shows how to predict fragment intensities with a **local** model instead of a model hosted on Koina.
The intensity model is a [DLOmix](https://github.com/wilhelm-lab/dlomix) pipeline saved to disk
(a folder containing `model.keras`, `dlomix_preprocessor.json` and `dlomix_inference_pipeline.json`).

- `models.dlomix_intensity` points at the local model folder and replaces `models.intensity`.
- Retention times are still predicted with the `irt` model via Koina. Use `"irt": "zero_irt"` to skip retention time
  prediction and run fully offline (retention time features will then not be informative).
- DLOmix runs single-threaded, so `numThreads` is set to 1 automatically.

Requires DLOmix to be installed (`pip install "oktoberfest[dlomix]"`).

## 1- Import necessary python packages

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from oktoberfest import predict as pr
from oktoberfest.data import Spectra
from oktoberfest.data.spectra import FragmentType
from oktoberfest.runner import run_job
from oktoberfest.utils import Config

## 2- Paths to the local model and the input data

In [ ]:
# HCD base model from the DLOmix publication (saved DLOmix InferencePipeline)
dlomix_model_path = "/cmnfs/proj/DLOmix/dlomix-publication/runs/01_base_models/hcd/pipeline"

# tomato example: MaxQuant search results and the matching mzML file
input_dir = Path("/cmnfs/home/w.gabriel/new_oktoberfest/notebook_runs/inputs/tomato")
search_results = str(input_dir / "msms.txt")
spectra = str(input_dir / "5407_GC4_063119_S00_U4_R1.mzML")

output_dir = Path("./out_local_model").resolve()
output_dir.mkdir(parents=True, exist_ok=True)

## 3- Predict intensities for a list of peptides

The predictor is created from a config that contains the local model, exactly as Oktoberfest does internally.

In [ ]:
prediction_config = {"type": "Rescoring", "output": str(output_dir), "models": {"dlomix_intensity": dlomix_model_path}}
with open(output_dir / "prediction_config.json", "w") as fp:
    json.dump(prediction_config, fp, indent=2)

config = Config()
config.read(output_dir / "prediction_config.json")
intensity_predictor = pr.Predictor.from_config(config, model_type="intensity")

In [ ]:
peptides = pd.DataFrame(
    {
        "MODIFIED_SEQUENCE": ["AAAC[UNIMOD:4]RFVQ", "LGEYGFQNALIVR", "VEGLM[UNIMOD:35]AGHK"],
        "COLLISION_ENERGY": [30, 30, 30],
        "PRECURSOR_CHARGE": [2, 2, 3],
        "FRAGMENTATION": ["HCD", "HCD", "HCD"],
    }
)
library = Spectra(obs=peptides, var=Spectra._gen_vars_df())
library.strings_to_categoricals()

intensity_predictor.predict_intensities(data=library)

In [ ]:
# 10 most intense predicted fragments per peptide
predicted = library.get_matrix(FragmentType.PRED).toarray()
top_fragments = {
    peptide: pd.Series(intensities, index=library.var_names).nlargest(10).round(3)
    for peptide, intensities in zip(peptides["MODIFIED_SEQUENCE"], predicted)
}
pd.DataFrame({peptide: list(zip(top.index, top.values)) for peptide, top in top_fragments.items()})

## 4- Rescoring with the local intensity model

Watch the log for `Using DLOmix intensity model`, which confirms that the intensities were predicted locally.

In [ ]:
task_config_local_rescoring = {
    "type": "Rescoring",
    "tag": "",
    "inputs": {
        "search_results": search_results,
        "search_results_type": "Maxquant",
        "spectra": spectra,
        "spectra_type": "mzml",
    },
    "output": str(output_dir / "rescoring"),
    "models": {
        "dlomix_intensity": dlomix_model_path,
        "irt": "Prosit_2019_irt",
    },
    "prediction_server": "koina.wilhelmlab.org:443",
    "ssl": True,
    "fdr_estimation_method": "percolator",
    "regressionMethod": "spline",
    "allFeatures": False,
    "massTolerance": 20,
    "unitMassTolerance": "ppm",
}
with open(output_dir / "local_rescoring_config.json", "w") as fp:
    json.dump(task_config_local_rescoring, fp, indent=2)

In [ ]:
run_job(output_dir / "local_rescoring_config.json")

## 5- Results

In [ ]:
fdr_dir = output_dir / "rescoring" / "results" / "percolator"
summary = {}
for search_type in ["original", "rescore"]:
    psms = pd.read_csv(fdr_dir / f"{search_type}.percolator.psms.txt", sep="\t")
    peptides_df = pd.read_csv(fdr_dir / f"{search_type}.percolator.peptides.txt", sep="\t")
    summary[search_type] = {
        "PSMs at 1% FDR": int((psms["q-value"] <= 0.01).sum()),
        "peptides at 1% FDR": int((peptides_df["q-value"] <= 0.01).sum()),
    }
pd.DataFrame(summary)